## Video Dubbing Pipeline
This section generates the English audio from the English translated transcript and append it to the video.
### 1. Engine Warming
- Get the system dependencies, download Kokoro Model Weights. Megatools is to download tools from Mega.nz.
- Get the python script that dubs the audio from the srt.
- Optional, if you wanna inspect/edit the script run this line in its own code cell: `%load https://raw.githubusercontent.com/aladinovitch/OpenDubber/main/autodub.py`

In [ ]:
# 1. Install system utilities
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg megatools
# 2. Completely strip existing ONNX packages to prevent CPU/GPU conflicts, as Kaggle env is pre-loaded with bunch of packages.
!pip uninstall -y -qq onnxruntime onnxruntime-gpu
# 3. Install Kokoro dependencies and kokoro-onnx without auto-installing CPU onnxruntime, then installs GPU ONNX runtime with CUDA support
!pip install -qq soundfile misaki[en]
!pip install -qq --no-deps kokoro-onnx
# Kaggle didn't yet use CUDA 13, when it does, replace the following section with this simple command: !pip install -qq onnxruntime-gpu
# 4. Reinstall onnxruntime-gpu specifically targeted for CUDA 12 (Kaggle T4 compatibility)
!pip install -qq onnxruntime-gpu \
  --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
# 5. Download model weights and voices (skips download if already local)
!wget -nc -q https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!wget -nc -q https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin
# 6. Get the python script to dub & srt cleaning
!curl -s -O https://raw.githubusercontent.com/aladinovitch/OpenDubber/main/autodub.py
!curl -s -O https://raw.githubusercontent.com/aladinovitch/OpenDubber/main/batchdub.py
!curl -s -O https://raw.githubusercontent.com/aladinovitch/OpenDubber/main/dashboard.py
!curl -s -O https://raw.githubusercontent.com/aladinovitch/OpenDubber/main/subfix.py
# 7. Preload CUDA libraries via PyTorch and verify GPU provider status
from dashboard import launch_dashboard
import torch
import onnxruntime as ort
providers = ort.get_available_providers()
isCuda = "✅ CUDA GPU acceleration is active." if "CUDAExecutionProvider" in providers else "⚠️ CUDA Provider not found. Running on CPU execution."
print(isCuda)

### 2. Dubbing Machine

In [ ]:
launch_dashboard()

### 3. Inspection Bay (optional)

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import subprocess

# Define base working directories dynamically
base_dir = Path("/kaggle/working")
work_dir = base_dir / dubdir
work_dir.mkdir(parents=True, exist_ok=True)

def preview_video_sample(video_path, start_time="00:00:00", duration="20", width=500):
    # Save temporary preview inside the base working directory dynamically
    sample_path = base_dir / "sample_preview.mp4"
    
    # Fast few seconds cut & lightweight encode for browser preview
    cmd = (
        f'ffmpeg -hide_banner -loglevel error -y -ss {start_time} -i "{video_path}" '
        f'-t {duration} -c:v libx264 -preset ultrafast -pix_fmt yuv420p -c:a aac "{sample_path}"'
    )
    subprocess.run(cmd, shell=True)

    # Read and encode
    mp4 = sample_path.read_bytes()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    
    display(HTML(f'''
        <video width="{width}" controls autoplay>
            <source src="{data_url}" type="video/mp4">
        </video>
    '''))

for output_mp4 in work_dir.iterdir():
    #if output_mp4.name.startswith("[En dub]") and output_mp4.suffix == ".mp4":
    if output_mp4.name.startswith("2v2") and output_mp4.suffix == ".mp4":
        print(f"🎥 Previewing: {output_mp4.name}")
        preview_video_sample(output_mp4, width=1000)

## Generate the transcript
This section is when no transcript is present in the YouTube video, use the following tools to generate subtitles in the foreign language
- `pytubefix` to download the audio from the video. You can another lib like pytubefix is this one gets blocked by YT.
- `whisper` to transcribe the text into and srt. `--model turbo` works well. `--model large-v3` for when heavy background noise, poor mic, overlapping dialogue and silent gaps.

In [ ]:
!pip install -qq pytubefix git+https://github.com/openai/whisper.git

### 1. Assets naming

In [ ]:
transcript_dubdir = "transcipt"
yt_url = "https://youtu.be/YOUTUBE_VIDEO"
extracted_audio_filename = "extracted_yt_audio_foreign_laguage.mp3"

# Prepare the paths
from pathlib import Path
work_dir = Path("/kaggle/working") / transcript_dubdir
work_dir.mkdir(parents=True, exist_ok=True)
extracted_audio = work_dir / extracted_audio_filename

### 2. Download the audion from YouTube

In [ ]:
from pytubefix import YouTube
yt = YouTube(yt_url)
audio_stream = yt.streams.filter(only_audio=True).first()
audio_stream.download(filename = extracted_audio)

### 3. Transcribe the audio into a subtitle

In [ ]:
!whisper "{extracted_audio}" --model turbo --language es --output_format srt

## Helpers section
### Subtitles fix
This section calls a python script to check the subtitles and clean it as needed.
```bash
!python subfix.py --srt "/kaggle/working/dubdir/subtitles.srt"
!python subfix.py --srt "/kaggle/working/dubdir/subtitles.srt" --out "/kaggle/working/dubdir/subtitles_cleand.srt"
!python subfix.py --help
```

In [ ]:
!python subfix.py --srt "/kaggle/working/dubdir/subtitles.srt"